In [2]:
# ============================== SETUP & LIBRARIES ==============================
import os
import sys
import time
import math
import random
import subprocess
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Dynamically install PyG dependencies if missing
try:
    import torch_geometric
except ImportError:
    print("Installing PyTorch Geometric and dependencies...")
    torch_ver = torch.__version__.split('+')[0]
    cuda_ver = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
    cmd = f"pip install --quiet torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver}.html"
    subprocess.run(cmd, shell=True, check=True)

from torchvision import models
from torchvision.datasets import ImageFolder
from torch_geometric.data import Data as GraphData, Batch
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool

import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# ============================== CONFIGURATION ==============================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ROOT_DIR = "/kaggle/input/datasets/abdelrahmantarekm/final-merged-dataset/final_dataset"
TRAIN_DIR = os.path.join(ROOT_DIR, "train")
VAL_DIR   = os.path.join(ROOT_DIR, "validation")
TEST_DIR  = os.path.join(ROOT_DIR, "test")

IMG_SIZE = 128
PATCH_SIZE = 16  # 128/16 = 8x8 grid (64 nodes)
BATCH_SIZE = 64
EPOCHS = 5
LR = 2e-4
PATIENCE = 5
NUM_WORKERS = 2
FUSION_DIM = 128
CHECKPOINT_PATH = "./best_cnn_gat_model.pt"

def set_seed(seed=SEED):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

set_seed()
print(f"Execution Device: {DEVICE}")

# ============================== SANITY CHECKS ==============================
def check_dataset_split(train_dir, val_dir, test_dir):
    assert os.path.exists(train_dir), f"Train directory not found: {train_dir}"
    assert os.path.exists(val_dir), f"Validation directory not found: {val_dir}"
    assert os.path.exists(test_dir), f"Test directory not found: {test_dir}"

    train_ds = ImageFolder(train_dir)
    val_ds = ImageFolder(val_dir)
    test_ds = ImageFolder(test_dir)

    print(f"Class Mapping: {train_ds.class_to_idx}")
    print(f"Dataset Counts -> Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

check_dataset_split(TRAIN_DIR, VAL_DIR, TEST_DIR)

# ============================== DATA AUGMENTATION ==============================
def get_train_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.95, 1.05), translate_percent=(-0.03, 0.03), rotate=(-8, 8), p=0.4),
        A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.03, p=0.4),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])

def get_eval_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])

# ============================== ENHANCED GRAPH FORMULATION ==============================
def image_to_graph(image_tensor, patch_size=PATCH_SIZE):
    c, h, w = image_tensor.shape
    grid_h = h // patch_size
    grid_w = w // patch_size

    patches = image_tensor.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
    patches = patches.permute(1, 2, 0, 3, 4).contiguous().reshape(-1, c, patch_size, patch_size)

    mean = patches.mean(dim=(2, 3))
    std  = patches.std(dim=(2, 3))
    amin = patches.amin(dim=(2, 3))
    amax = patches.amax(dim=(2, 3))

    ys, xs = torch.meshgrid(
        torch.arange(grid_h, dtype=torch.float32),
        torch.arange(grid_w, dtype=torch.float32),
        indexing='ij'
    )
    pos_y = (ys.flatten() / grid_h).unsqueeze(1)
    pos_x = (xs.flatten() / grid_w).unsqueeze(1)

    x = torch.cat([mean, std, amin, amax, pos_y, pos_x], dim=1)

    edge_list = []
    for i in range(grid_h):
        for j in range(grid_w):
            idx = i * grid_w + j
            if i > 0: edge_list.append([idx, (i - 1) * grid_w + j])
            if i < grid_h - 1: edge_list.append([idx, (i + 1) * grid_w + j])
            if j > 0: edge_list.append([idx, i * grid_w + (j - 1)])
            if j < grid_w - 1: edge_list.append([idx, i * grid_w + (j + 1)])

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    return GraphData(x=x, edge_index=edge_index)

# ============================== DATASET CLASS ==============================
class SplitImageGraphDataset(Dataset):
    def __init__(self, split_dir, transform):
        self.folder = ImageFolder(split_dir)
        self.transform = transform

    def __len__(self):
        return len(self.folder.samples)

    def __getitem__(self, idx):
        path, label = self.folder.samples[idx]
        img = Image.open(path).convert("RGB")
        img = np.array(img)
        img = self.transform(image=img)["image"]
        graph = image_to_graph(img)
        return img, graph, torch.tensor(label, dtype=torch.long)

train_ds = SplitImageGraphDataset(TRAIN_DIR, get_train_transforms())
val_ds   = SplitImageGraphDataset(VAL_DIR, get_eval_transforms())
test_ds  = SplitImageGraphDataset(TEST_DIR, get_eval_transforms())

def custom_collate(batch):
    imgs = torch.stack([x[0] for x in batch])
    graphs = Batch.from_data_list([x[1] for x in batch])
    labels = torch.stack([x[2] for x in batch])
    return imgs, graphs, labels

pin_mem = DEVICE == "cuda"

# FIX: Set drop_last=True on train_loader to prevent single-sample batches from breaking BatchNorm
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=NUM_WORKERS, pin_memory=pin_mem, 
    drop_last=True, collate_fn=custom_collate
)
val_loader   = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=NUM_WORKERS, pin_memory=pin_mem, 
    drop_last=False, collate_fn=custom_collate
)
test_loader  = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=NUM_WORKERS, pin_memory=pin_mem, 
    drop_last=False, collate_fn=custom_collate
)

# ============================== MODEL ARCHITECTURE ==============================
class CNNBranch(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        for p in backbone.parameters():
            p.requires_grad = False
        for p in backbone.layer3.parameters():
            p.requires_grad = True
        for p in backbone.layer4.parameters():
            p.requires_grad = True

        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(512, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

class GATBranch(nn.Module):
    def __init__(self, in_dim=14, hidden=64, out_dim=128, dropout=0.30):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden, heads=2, concat=True, dropout=dropout)
        self.conv2 = GATConv(hidden * 2, hidden, heads=1, concat=False, dropout=dropout)
        self.norm1 = nn.LayerNorm(hidden * 2)
        self.norm2 = nn.LayerNorm(hidden)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )

    def forward(self, batch_graph):
        x = self.conv1(batch_graph.x, batch_graph.edge_index)
        x = self.norm1(F.elu(x))
        x = self.drop(x)
        x = self.conv2(x, batch_graph.edge_index)
        x = self.norm2(F.elu(x))

        x_mean = global_mean_pool(x, batch_graph.batch)
        x_max = global_max_pool(x, batch_graph.batch)
        x = torch.cat([x_mean, x_max], dim=1)
        return self.head(x)

class GatedFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, cnn_feat, gat_feat):
        g = self.gate(torch.cat([cnn_feat, gat_feat], dim=1))
        fused = g * cnn_feat + (1.0 - g) * gat_feat
        fused = self.norm(fused)

        diff = torch.abs(cnn_feat - gat_feat)
        prod = cnn_feat * gat_feat
        return torch.cat([fused, diff, prod], dim=1)

class CNN_GAT_Fusion(nn.Module):
    def __init__(self, dim=FUSION_DIM, num_classes=2):
        super().__init__()
        self.cnn = CNNBranch(out_dim=dim)
        self.gat = GATBranch(out_dim=dim)
        self.fusion = GatedFusion(dim)
        self.classifier = nn.Sequential(
            nn.Linear(dim * 3, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(128, num_classes),
        )

    def forward(self, imgs, batch_graph):
        cnn_feat = self.cnn(imgs)
        gat_feat = self.gat(batch_graph)
        fused = self.fusion(cnn_feat, gat_feat)
        return self.classifier(fused)

# ============================== OPTIMIZATION & TRAINING ==============================
model = CNN_GAT_Fusion().to(DEVICE)

train_counts = Counter(train_ds.folder.targets)
weights = torch.tensor([1.0 / train_counts[c] for c in range(2)], dtype=torch.float32)
weights = (weights / weights.sum() * 2).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

def run_epoch(model, loader, is_train=True):
    model.train() if is_train else model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    y_true, y_pred, y_prob = [], [], []

    pbar = tqdm(loader, leave=False, desc="Training" if is_train else "Evaluating")
    for imgs, graphs, labels in pbar:
        # Safety check for single sample batch during training
        if is_train and imgs.size(0) <= 1:
            continue

        imgs, graphs, labels = imgs.to(DEVICE, non_blocking=True), graphs.to(DEVICE), labels.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            logits = model(imgs, graphs)
            loss = criterion(logits, labels)

        if is_train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        probs = F.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs[:, 1].detach().cpu().numpy())

    return total_loss / total_samples, total_correct / total_samples, np.array(y_true), np.array(y_pred), np.array(y_prob)

# ============================== EXECUTION LOOP ==============================
print("\nStarting Training Execution Loop...")
best_val_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc, _, _, _ = run_epoch(model, train_loader, is_train=True)
    val_loss, val_acc, _, _, _ = run_epoch(model, val_loader, is_train=False)

    print(f"Epoch {epoch+1:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    scheduler.step(val_acc)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT_PATH)

# ============================== COMPREHENSIVE EVALUATION ==============================
print("\nLoading Best Model Checkpoint for Metric Evaluation...")
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

start_time = time.time()
test_loss, test_acc, y_true, y_pred, y_prob = run_epoch(model, test_loader, is_train=False)
total_time = time.time() - start_time
latency_per_frame = (total_time / len(test_ds)) * 1000

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)
auc  = roc_auc_score(y_true, y_prob)
cm   = confusion_matrix(y_true, y_pred)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "="*20 + " EXPERIMENTAL TEST METRICS " + "="*20)
print(f"Accuracy         : {acc:.4f}")
print(f"Precision        : {prec:.4f}")
print(f"Recall           : {rec:.4f}")
print(f"F1-Score         : {f1:.4f}")
print(f"AUC-ROC          : {auc:.4f}")
print(f"Inference Latency: {latency_per_frame:.2f} ms / frame")
print(f"Trainable Params : {trainable_params / 1e6:.2f} Million")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Fake", "Real"], zero_division=0))

Execution Device: cuda
Class Mapping: {'fake': 0, 'real': 1}
Dataset Counts -> Train: 270209 | Val: 66939 | Test: 40261

Starting Training Execution Loop...


Epoch 01/05 | Train Loss: 0.2208 Acc: 0.9045 | Val Loss: 0.1468 Acc: 0.9414


Epoch 02/05 | Train Loss: 0.1280 Acc: 0.9492 | Val Loss: 0.1027 Acc: 0.9612


Epoch 03/05 | Train Loss: 0.0980 Acc: 0.9617 | Val Loss: 0.1091 Acc: 0.9589


Epoch 05/05 | Train Loss: 0.0684 Acc: 0.9739 | Val Loss: 0.0856 Acc: 0.9700

Loading Best Model Checkpoint for Metric Evaluation...



==================== EXPERIMENTAL TEST METRICS ====================
Accuracy         : 0.9281
Precision        : 0.9458
Recall           : 0.8968
F1-Score         : 0.9206
AUC-ROC          : 0.9800
Inference Latency: 4.12 ms / frame
Trainable Params : 10.67 Million

Confusion Matrix:
[[20567   963]
 [ 1933 16798]]

Classification Report:
              precision    recall  f1-score   support

        Fake       0.91      0.96      0.93     21530
        Real       0.95      0.90      0.92     18731

    accuracy                           0.93     40261
   macro avg       0.93      0.93      0.93     40261
weighted avg       0.93      0.93      0.93     40261



In [4]:
# ============================== SETUP & LIBRARIES ==============================
import os
import sys
import time
import math
import random
import subprocess
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Dynamically install PyG dependencies if missing
try:
    import torch_geometric
except ImportError:
    print("Installing PyTorch Geometric and dependencies...")
    torch_ver = torch.__version__.split('+')[0]
    cuda_ver = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
    cmd = f"pip install --quiet torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver}.html"
    subprocess.run(cmd, shell=True, check=True)

from torchvision import models
from torchvision.datasets import ImageFolder
from torch_geometric.data import Data as GraphData, Batch
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool

import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# ============================== CONFIGURATION ==============================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ROOT_DIR = "/kaggle/input/datasets/abdelrahmantarekm/final-merged-dataset/final_dataset"
TRAIN_DIR = os.path.join(ROOT_DIR, "train")
VAL_DIR   = os.path.join(ROOT_DIR, "validation")
TEST_DIR  = os.path.join(ROOT_DIR, "test")

IMG_SIZE = 128
PATCH_SIZE = 16  # 128/16 = 8x8 grid (64 nodes)
BATCH_SIZE = 64
EPOCHS = 5
LR = 2e-4
PATIENCE = 5
NUM_WORKERS = 2
FUSION_DIM = 128
CHECKPOINT_PATH = "./best_cnn_gat_model.pt"

def set_seed(seed=SEED):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

set_seed()
print(f"Execution Device: {DEVICE}")

# ============================== SANITY CHECKS ==============================
def check_dataset_split(train_dir, val_dir, test_dir):
    assert os.path.exists(train_dir), f"Train directory not found: {train_dir}"
    assert os.path.exists(val_dir), f"Validation directory not found: {val_dir}"
    assert os.path.exists(test_dir), f"Test directory not found: {test_dir}"

    train_ds = ImageFolder(train_dir)
    val_ds = ImageFolder(val_dir)
    test_ds = ImageFolder(test_dir)

    print(f"Class Mapping: {train_ds.class_to_idx}")
    print(f"Dataset Counts -> Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

check_dataset_split(TRAIN_DIR, VAL_DIR, TEST_DIR)

# ============================== DATA AUGMENTATION ==============================
def get_train_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.95, 1.05), translate_percent=(-0.03, 0.03), rotate=(-8, 8), p=0.4),
        A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.03, p=0.4),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])

def get_eval_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])

# ============================== ENHANCED GRAPH FORMULATION ==============================
def image_to_graph(image_tensor, patch_size=PATCH_SIZE):
    c, h, w = image_tensor.shape
    grid_h = h // patch_size
    grid_w = w // patch_size

    patches = image_tensor.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
    patches = patches.permute(1, 2, 0, 3, 4).contiguous().reshape(-1, c, patch_size, patch_size)

    mean = patches.mean(dim=(2, 3))
    std  = patches.std(dim=(2, 3))
    amin = patches.amin(dim=(2, 3))
    amax = patches.amax(dim=(2, 3))

    ys, xs = torch.meshgrid(
        torch.arange(grid_h, dtype=torch.float32),
        torch.arange(grid_w, dtype=torch.float32),
        indexing='ij'
    )
    pos_y = (ys.flatten() / grid_h).unsqueeze(1)
    pos_x = (xs.flatten() / grid_w).unsqueeze(1)

    x = torch.cat([mean, std, amin, amax, pos_y, pos_x], dim=1)

    edge_list = []
    for i in range(grid_h):
        for j in range(grid_w):
            idx = i * grid_w + j
            if i > 0: edge_list.append([idx, (i - 1) * grid_w + j])
            if i < grid_h - 1: edge_list.append([idx, (i + 1) * grid_w + j])
            if j > 0: edge_list.append([idx, i * grid_w + (j - 1)])
            if j < grid_w - 1: edge_list.append([idx, i * grid_w + (j + 1)])

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    return GraphData(x=x, edge_index=edge_index)

# ============================== DATASET CLASS ==============================
class SplitImageGraphDataset(Dataset):
    def __init__(self, split_dir, transform):
        self.folder = ImageFolder(split_dir)
        self.transform = transform

    def __len__(self):
        return len(self.folder.samples)

    def __getitem__(self, idx):
        path, label = self.folder.samples[idx]
        img = Image.open(path).convert("RGB")
        img = np.array(img)
        img = self.transform(image=img)["image"]
        graph = image_to_graph(img)
        return img, graph, torch.tensor(label, dtype=torch.long)

train_ds = SplitImageGraphDataset(TRAIN_DIR, get_train_transforms())
val_ds   = SplitImageGraphDataset(VAL_DIR, get_eval_transforms())
test_ds  = SplitImageGraphDataset(TEST_DIR, get_eval_transforms())

def custom_collate(batch):
    imgs = torch.stack([x[0] for x in batch])
    graphs = Batch.from_data_list([x[1] for x in batch])
    labels = torch.stack([x[2] for x in batch])
    return imgs, graphs, labels

pin_mem = DEVICE == "cuda"

# FIX: Set drop_last=True on train_loader to prevent single-sample batches from breaking BatchNorm
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=NUM_WORKERS, pin_memory=pin_mem, 
    drop_last=True, collate_fn=custom_collate
)
val_loader   = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=NUM_WORKERS, pin_memory=pin_mem, 
    drop_last=False, collate_fn=custom_collate
)
test_loader  = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=NUM_WORKERS, pin_memory=pin_mem, 
    drop_last=False, collate_fn=custom_collate
)

# ============================== MODEL ARCHITECTURE ==============================
class CNNBranch(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        for p in backbone.parameters():
            p.requires_grad = False
        for p in backbone.layer3.parameters():
            p.requires_grad = True
        for p in backbone.layer4.parameters():
            p.requires_grad = True

        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(512, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

class GATBranch(nn.Module):
    def __init__(self, in_dim=14, hidden=64, out_dim=128, dropout=0.30):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden, heads=2, concat=True, dropout=dropout)
        self.conv2 = GATConv(hidden * 2, hidden, heads=1, concat=False, dropout=dropout)
        self.norm1 = nn.LayerNorm(hidden * 2)
        self.norm2 = nn.LayerNorm(hidden)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )

    def forward(self, batch_graph):
        x = self.conv1(batch_graph.x, batch_graph.edge_index)
        x = self.norm1(F.elu(x))
        x = self.drop(x)
        x = self.conv2(x, batch_graph.edge_index)
        x = self.norm2(F.elu(x))

        x_mean = global_mean_pool(x, batch_graph.batch)
        x_max = global_max_pool(x, batch_graph.batch)
        x = torch.cat([x_mean, x_max], dim=1)
        return self.head(x)

class GatedFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, cnn_feat, gat_feat):
        g = self.gate(torch.cat([cnn_feat, gat_feat], dim=1))
        fused = g * cnn_feat + (1.0 - g) * gat_feat
        fused = self.norm(fused)

        diff = torch.abs(cnn_feat - gat_feat)
        prod = cnn_feat * gat_feat
        return torch.cat([fused, diff, prod], dim=1)

class CNN_GAT_Fusion(nn.Module):
    def __init__(self, dim=FUSION_DIM, num_classes=2):
        super().__init__()
        self.cnn = CNNBranch(out_dim=dim)
        self.gat = GATBranch(out_dim=dim)
        self.fusion = GatedFusion(dim)
        self.classifier = nn.Sequential(
            nn.Linear(dim * 3, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(128, num_classes),
        )

    def forward(self, imgs, batch_graph):
        cnn_feat = self.cnn(imgs)
        gat_feat = self.gat(batch_graph)
        fused = self.fusion(cnn_feat, gat_feat)
        return self.classifier(fused)


Execution Device: cuda
Class Mapping: {'fake': 0, 'real': 1}
Dataset Counts -> Train: 270209 | Val: 66939 | Test: 40261


In [6]:

# ============================== OPTIMIZATION & TRAINING ==============================
model = CNN_GAT_Fusion().to(DEVICE)

train_counts = Counter(train_ds.folder.targets)
weights = torch.tensor([1.0 / train_counts[c] for c in range(2)], dtype=torch.float32)
weights = (weights / weights.sum() * 2).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

def run_epoch(model, loader, is_train=True):
    model.train() if is_train else model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    y_true, y_pred, y_prob = [], [], []

    pbar = tqdm(loader, leave=False, desc="Training" if is_train else "Evaluating")
    for imgs, graphs, labels in pbar:
        # Safety check for single sample batch during training
        if is_train and imgs.size(0) <= 1:
            continue

        imgs, graphs, labels = imgs.to(DEVICE, non_blocking=True), graphs.to(DEVICE), labels.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            logits = model(imgs, graphs)
            loss = criterion(logits, labels)

        if is_train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        probs = F.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs[:, 1].detach().cpu().numpy())

    return total_loss / total_samples, total_correct / total_samples, np.array(y_true), np.array(y_pred), np.array(y_prob)


In [7]:
# ============================== FIXED ABLATION MODELS ==============================
class CNNOnlyModel(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.cnn = CNNBranch(out_dim=128)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(64, num_classes)
        )
    def forward(self, imgs, batch_graph):
        return self.classifier(self.cnn(imgs))

class GATOnlyModel(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.gat = GATBranch(out_dim=128)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(64, num_classes)
        )
    def forward(self, imgs, batch_graph):
        return self.classifier(self.gat(batch_graph))

# Updated epoch function accepting custom optimizer
def run_ablation_epoch(model, loader, opt, scaler, is_train=True):
    model.train() if is_train else model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    y_true, y_pred, y_prob = [], [], []

    pbar = tqdm(loader, leave=False, desc="Training" if is_train else "Evaluating")
    for imgs, graphs, labels in pbar:
        if is_train and imgs.size(0) <= 1:
            continue

        imgs, graphs, labels = imgs.to(DEVICE, non_blocking=True), graphs.to(DEVICE), labels.to(DEVICE, non_blocking=True)

        if is_train:
            opt.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            logits = model(imgs, graphs)
            loss = criterion(logits, labels)

        if is_train:
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

        probs = F.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs[:, 1].detach().cpu().numpy())

    return total_loss / total_samples, total_correct / total_samples, np.array(y_true), np.array(y_pred), np.array(y_prob)

def evaluate_ablation_variant(variant_model, model_name):
    print(f"\n--- Running Ablation Experiment: {model_name} ---")
    variant_model = variant_model.to(DEVICE)
    opt = torch.optim.AdamW(variant_model.parameters(), lr=LR, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    
    # 3-epoch evaluation for ablation comparison
    for ep in range(3):
        print(f"Epoch {ep+1}/3...")
        run_ablation_epoch(variant_model, train_loader, opt, scaler, is_train=True)
    
    _, _, y_t, y_p, y_pr = run_ablation_epoch(variant_model, test_loader, opt, scaler, is_train=False)
    
    acc = accuracy_score(y_t, y_p)
    f1  = f1_score(y_t, y_p, zero_division=0)
    auc = roc_auc_score(y_t, y_pr)
    return acc, f1, auc

# Execute Ablation Tests
cnn_acc, cnn_f1, cnn_auc = evaluate_ablation_variant(CNNOnlyModel(), "CNN-Branch Only")
gat_acc, gat_f1, gat_auc = evaluate_ablation_variant(GATOnlyModel(), "GAT-Branch Only")

print("\n" + "="*20 + " FINAL ABLATION STUDY TABLE " + "="*20)
print(f"CNN Branch Only      | Accuracy: {cnn_acc:.4f} | F1: {cnn_f1:.4f} | AUC: {cnn_auc:.4f}")
print(f"GAT Branch Only      | Accuracy: {gat_acc:.4f} | F1: {gat_f1:.4f} | AUC: {gat_auc:.4f}")
print(f"Proposed CNN-GAT     | Accuracy: 0.9281 | F1: 0.9206 | AUC: 0.9800")


--- Running Ablation Experiment: CNN-Branch Only ---
Epoch 1/3...


Epoch 2/3...


Epoch 3/3...



--- Running Ablation Experiment: GAT-Branch Only ---
Epoch 1/3...


Epoch 2/3...


Epoch 3/3...



==================== FINAL ABLATION STUDY TABLE ====================
CNN Branch Only      | Accuracy: 0.9222 | F1: 0.9145 | AUC: 0.9776
GAT Branch Only      | Accuracy: 0.6150 | F1: 0.4915 | AUC: 0.6930
Proposed CNN-GAT     | Accuracy: 0.9281 | F1: 0.9206 | AUC: 0.9800


In [8]:
import matplotlib.pyplot as plt

# Load trained main fusion model
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

# Extract single batch from test loader
sample_imgs, sample_graphs, _ = next(iter(test_loader))
sample_imgs, sample_graphs = sample_imgs.to(DEVICE), sample_graphs.to(DEVICE)

with torch.no_grad():
    cnn_f = model.cnn(sample_imgs)
    gat_f = model.gat(sample_graphs)
    # Compute gate activation values
    gate_values = model.fusion.gate(torch.cat([cnn_f, gat_f], dim=1)).cpu().numpy()

# Plot gating weights across feature indices
plt.figure(figsize=(10, 4))
plt.plot(gate_values[0], label="Sample Image Feature Allocation", color="tab:blue", linewidth=1.5)
plt.axhline(y=0.5, color='r', linestyle='--', label='Equal Weight Boundary (0.5)')
plt.xlabel("Feature Index")
plt.ylabel("Gate Weight (1.0 = CNN, 0.0 = GAT)")
plt.title("Adaptive Gated Fusion Weight Distribution")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("fusion_gate_weights.png", dpi=300, bbox_inches='tight')
plt.show()
print("Successfully generated and saved 'fusion_gate_weights.png'!")

FileNotFoundError: [Errno 2] No such file or directory: './best_cnn_gat_model.pt'